In [1]:
import pandas as pd

Data Loading and info

In [2]:
INPUT_PATH = '/Users/david/Documents/data/jusbrasil/manual/with_body/manual_full_2026_com_body.parquet'

jusbrasil_df = pd.read_parquet(INPUT_PATH)

In [3]:
print("Total of documents: ",jusbrasil_df.shape[0])
print("Total of queries: ",jusbrasil_df["query"].nunique())

Total of documents:  18685
Total of queries:  1089


In [10]:
counts = jusbrasil_df["query"].value_counts()
print(counts.mean())

17.157943067033976


In [4]:
jusbrasil_df

,query,query_category,query_tokens,document,relevance,bm25_body,bm25_copy_context,bm25_copy_context_exact_phrase_match,bm25_expanded_copy_context,bm25_highlight,...,bm25_theme_phrase_match,is_boosted_theme_number,bm25_plain_facts,bm25_plain_facts_exact_phrase_match,bm25_plain_facts_phrase_match,bm25_body_intervals,bm25_highlight_intervals,title,court,body
0,ação anulatória e ação rescisória,torso,acao anulatoria e acao rescisoria,2285767469,0,32.291805,5.857748,NaN,NaN,24.512104,...,NaN,0.0,18.226200,NaN,NaN,0.750000,0.500000,AÇÃO RESCISÓRIA 8064350920224050000,TRF_5,PROCESSO Nº: 0806435-09.2022.4.05.0000 - AÇÃO ...
1,ação anulatória e ação rescisória,torso,acao anulatoria e acao rescisoria,935431201,1,32.595516,NaN,NaN,NaN,31.083240,...,NaN,0.0,12.231331,NaN,NaN,0.477345,0.100000,REFERENDO NA MEDIDA CAUTELAR NA AÇÃO RESCISÓRI...,STF,Supremo Tribunal Federal EmentaeAcórdão Inteir...
2,ação anulatória e ação rescisória,torso,acao anulatoria e acao rescisoria,583723433,1,25.257212,NaN,NaN,NaN,31.238258,...,NaN,0.0,NaN,NaN,NaN,0.021978,NaN,Ação Rescisória: AR 70072211626 RS,TJ_RS,@ (PROCESSO ELETRÔNICO) PRLF Nº 70072211626 (N...
3,ação anulatória e ação rescisória,torso,acao anulatoria e acao rescisoria,1990411642,1,34.785557,33.111893,3.309932,3.39329,22.770103,...,NaN,0.0,19.659252,NaN,NaN,0.584871,NaN,RECURSO ESPECIAL: REsp 2064264 PA 2023/0113693-5,STJ,RECURSO ESPECIAL Nº 2064264 - PA (2023/0113693...
4,ação anulatória e ação rescisória,torso,acao anulatoria e acao rescisoria,1209874412,1,28.115580,NaN,NaN,NaN,NaN,...,NaN,0.0,NaN,NaN,NaN,0.558360,NaN,AÇÃO RESCISÓRIA: AR 6996 DF 2021/0032355-3,STJ,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18680,violência doméstica vias de fato,tail,violencia domestica vias de fato,3600188628,1,40.033134,NaN,NaN,NaN,NaN,...,NaN,0.0,NaN,NaN,NaN,0.066667,NaN,Ação Penal - Procedimento Ordinário 10689-55.2...,TJ_CE,SENTENÇA Processo n.º: 0010689-55.2019.8.06.01...
18681,violência doméstica vias de fato,tail,violencia domestica vias de fato,2548888232,2,44.378605,24.554830,4.624367,NaN,31.617857,...,NaN,0.0,35.699140,NaN,NaN,0.637827,0.200000,AGRAVO REGIMENTAL NO AGRAVO EM RECURSO ESPECIA...,STJ,AgRg no AGRAVO EM RECURSO ESPECIAL Nº 2429109 ...
18682,violência doméstica vias de fato,tail,violencia domestica vias de fato,5137597575,2,48.118200,14.041665,NaN,NaN,45.314266,...,NaN,0.0,35.790170,NaN,NaN,0.295465,0.212527,Apelação Criminal 229686820248130479,TJ_MG,EMENTA: DIREITO PENAL E PROCESSUAL PENAL. APEL...
18683,violência doméstica vias de fato,tail,violencia domestica vias de fato,1659993560,1,27.854807,21.971668,NaN,NaN,20.983520,...,NaN,0.0,19.023026,NaN,NaN,NaN,NaN,AGRAVO REGIMENTAL NO HABEAS CORPUS: AgRg no HC...,STJ,AgRg no HABEAS CORPUS Nº 697993 - ES (2021/031...


In [22]:
print("Max value: ", jusbrasil_df['body'].str.len().max())
print("Mean value: ", jusbrasil_df['body'].str.len().mean())
print("Median value: ", jusbrasil_df['body'].str.len().median())


Max value:  646545.0
Mean value:  20712.636264555727
Median value:  14503.0


Query category (head / torso / tail)

Raw monthly exports include `query_category`. The parquet does not. There is also a ready mapping at `manual/backup/query_categories.csv`.

Join key: `query` after `strip()` — 25 parquet queries only fail an exact match because of leading/trailing spaces in the raw files. After strip, **all 1089 parquet queries** get a category.

Extra labels besides head/torso/tail: `navigational` and `goldenset`.

One conflict in raw: `multa por litigância de má-fé. excessiva` is `torso` in 2025.10 and `head` in 2025.12. The mapping CSV keeps the later revision (`head`).

In [15]:
from pathlib import Path

RAW_DIR = Path("/Users/david/Documents/data/jusbrasil/manual/backup/raw")
QUERY_CATEGORY_PATH = Path("/Users/david/Documents/data/jusbrasil/manual/backup/query_categories.csv")


def norm_query(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()


raw_df = pd.concat(
    (pd.read_csv(path, usecols=["query", "query_category", "revision"]) for path in sorted(RAW_DIR.glob("*.csv"))),
    ignore_index=True,
)
raw_df["query_norm"] = norm_query(raw_df["query"])

query_categories = pd.read_csv(QUERY_CATEGORY_PATH)
query_categories["query_norm"] = norm_query(query_categories["query"])

print("Raw rows:", len(raw_df))
print("Raw unique queries:", raw_df["query_norm"].nunique())
print("Mapping unique queries:", query_categories["query_norm"].nunique())
print("Raw category mix:")
print(raw_df.drop_duplicates("query_norm")["query_category"].value_counts())

Raw rows: 20709
Raw unique queries: 1159
Mapping unique queries: 1159
Raw category mix:
query_category
tail            534
torso           330
head            211
navigational     44
goldenset        40
Name: count, dtype: int64


In [ ]:
parquet_queries = jusbrasil_df[["query"]].drop_duplicates().copy()
parquet_queries["query_norm"] = norm_query(parquet_queries["query"])

parquet_n = parquet_queries["query_norm"].nunique()
in_raw = parquet_queries["query_norm"].isin(raw_df["query_norm"])
in_map = parquet_queries["query_norm"].isin(query_categories["query_norm"])

print(f"Parquet queries: {parquet_n}")
print(f"Recovered from raw CSVs: {in_raw.sum()} / {parquet_n} ({in_raw.mean():.1%})")
print(f"Recovered from query_categories.csv: {in_map.sum()} / {parquet_n} ({in_map.mean():.1%})")
print(f"Raw queries not in parquet: {len(set(raw_df['query_norm']) - set(parquet_queries['query_norm']))}")
print(f"Mapping queries not in parquet: {len(set(query_categories['query_norm']) - set(parquet_queries['query_norm']))}")

conflicts = (
    raw_df.groupby("query_norm")["query_category"]
    .nunique()
    .loc[lambda s: s > 1]
)
print(f"Queries with conflicting categories in raw: {len(conflicts)}")
if len(conflicts):
    display(raw_df.loc[raw_df["query_norm"].isin(conflicts.index), ["query", "query_category", "revision"]].drop_duplicates())

Parquet queries: 1089
Recovered from raw CSVs: 1089 / 1089 (100.0%)
Recovered from query_categories.csv: 1089 / 1089 (100.0%)
Raw queries not in parquet: 70
Mapping queries not in parquet: 70
Queries with conflicting categories in raw: 1


,query,query_category,revision
1123,multa por litigância de má-fé. excessiva,torso,2025.10
4207,multa por litigância de má-fé. excessiva,head,2025.12


In [ ]:
category_map = query_categories[["query_norm", "query_category"]].drop_duplicates("query_norm")

jusbrasil_df = jusbrasil_df.copy()
if "query_category" in jusbrasil_df.columns:
    jusbrasil_df = jusbrasil_df.drop(columns=["query_category"])

jusbrasil_df["query_norm"] = norm_query(jusbrasil_df["query"])
jusbrasil_df = jusbrasil_df.merge(category_map, on="query_norm", how="left")
jusbrasil_df = jusbrasil_df.drop(columns="query_norm")

cols = [c for c in jusbrasil_df.columns if c != "query_category"]
cols.insert(cols.index("query") + 1, "query_category")
jusbrasil_df = jusbrasil_df[cols]

print("Rows missing query_category:", jusbrasil_df["query_category"].isna().sum())
print("Documents by category:")
print(jusbrasil_df["query_category"].value_counts())
print("Queries by category:")
print(jusbrasil_df.drop_duplicates("query")["query_category"].value_counts())

Rows missing query_category: 0
Documents by category:
query_category
tail            8424
torso           5450
head            3510
navigational     664
goldenset        637
Name: count, dtype: int64
Queries by category:
query_category
tail            497
torso           313
head            201
navigational     41
goldenset        37
Name: count, dtype: int64


In [ ]:
# Already written to INPUT_PATH. Re-run this cell only if you want to overwrite again.
OUTPUT_PATH = INPUT_PATH
jusbrasil_df.to_parquet(OUTPUT_PATH, index=False)
print("Wrote", OUTPUT_PATH)
print("Columns:", list(jusbrasil_df.columns))

Wrote /Users/david/Documents/data/jusbrasil/manual/with_body/manual_full_2026_com_body.parquet
Columns: ['query', 'query_category', 'query_tokens', 'document', 'relevance', 'bm25_body', 'bm25_copy_context', 'bm25_copy_context_exact_phrase_match', 'bm25_expanded_copy_context', 'bm25_highlight', 'bm25_highlight_exact_phrase_match', 'bm25_highlight_phrase_match', 'bm25_metadata', 'bm25_title', 'is_boosted_court', 'authority', 'recency', 'is_mandatory_precedent', 'is_decision_merit', 'is_decision_admissibility', 'juris_type', 'guidance_status', 'summary_status', 'thesis_type', 'court_type', 'degree', 'bm25_thesis', 'bm25_thesis_exact_phrase_match', 'bm25_thesis_phrase_match', 'bm25_theme', 'bm25_theme_exact_phrase_match', 'bm25_theme_phrase_match', 'is_boosted_theme_number', 'bm25_plain_facts', 'bm25_plain_facts_exact_phrase_match', 'bm25_plain_facts_phrase_match', 'bm25_body_intervals', 'bm25_highlight_intervals', 'title', 'court', 'body']
